In [ ]:
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# ======================
# Parameters
# ======================

N        = 100          # number of particles
Lx       = 1.0          # box width
Ly       = 1.0          # box height
dt       = 0.005        # time step
k        = 0.01         # repulsion strength
eps      = 0.02         # softening (avoids 1/r singularity)
r_cut    = 0.15         # interaction cutoff radius

# ======================
# Initial Conditions
# ======================

rng = np.random.default_rng(seed=42)

x  = rng.uniform(0, Lx, N)
y  = rng.uniform(0, Ly, N)
vx = rng.uniform(-0.2,  0.2, N)
vy = rng.uniform(-0.2,  0.2, N)

# ======================
# Force Calculation
# ======================

def compute_acceleration(x, y):
    """
    Vectorized repulsive force via cKDTree neighbor queries.

    For each pair (i, j) within r_cut:
        vec_r  = pos_i - pos_j          (points from j → i  ⟹  repulsive)
        r2     = |vec_r|² + eps²        (softened)
        force  = k / r2                 (inverse-square magnitude)
        F_vec  = force * vec_r / |vec_r| (unit vector along vec_r)

    Newton's 3rd law: F on j = −F on i, so we do one pass over pairs.
    Any attractive component is absent because force magnitude > 0 always
    and the direction always points away from j.
    """
    pos  = np.column_stack((x, y))
    tree = cKDTree(pos)

    # All pairs within cutoff (each pair returned once with i < j)
    pairs = tree.query_pairs(r_cut, output_type='ndarray')  # shape (M, 2)

    ax_arr = np.zeros(N)
    ay_arr = np.zeros(N)

    if len(pairs) == 0:
        return ax_arr, ay_arr

    i_idx = pairs[:, 0]
    j_idx = pairs[:, 1]

    # Signed displacement  (i − j)  →  points from j to i  →  repulsive
    dx = x[i_idx] - x[j_idx]
    dy = y[i_idx] - y[j_idx]

    r2  = dx*dx + dy*dy + eps**2      # softened squared distance
    r   = np.sqrt(r2)
    mag = k / r2                       # inverse-square force magnitude

    fx = mag * dx / r                  # x-component of force on i
    fy = mag * dy / r                  # y-component of force on i

    # Accumulate: Newton's 3rd law gives equal & opposite force on j
    np.add.at(ax_arr, i_idx,  fx)
    np.add.at(ax_arr, j_idx, -fx)
    np.add.at(ay_arr, i_idx,  fy)
    np.add.at(ay_arr, j_idx, -fy)

    return ax_arr, ay_arr

# ======================
# Initial Acceleration
# ======================

ax_arr, ay_arr = compute_acceleration(x, y)

# ======================
# Plot Setup
# ======================

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)
ax.set_aspect('equal')
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')
ax.tick_params(colors='gray')
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

scat = ax.scatter(x, y, s=30, color='#5DCAA5', alpha=0.85, linewidths=0)

# ======================
# Time Integration
# ======================

def update(frame):
    global x, y, vx, vy, ax_arr, ay_arr

    # --- Save old acceleration ---
    ax_old = ax_arr.copy()
    ay_old = ay_arr.copy()

    # --- Position update (Verlet) ---
    x += vx * dt + 0.5 * ax_old * dt**2
    y += vy * dt + 0.5 * ay_old * dt**2

    # --- Wall reflections ---
    hit_x = (x < 0) | (x > Lx)
    hit_y = (y < 0) | (y > Ly)
    vx[hit_x] *= -1
    vy[hit_y] *= -1
    x = np.clip(x, 0, Lx)
    y = np.clip(y, 0, Ly)

    # --- New acceleration ---
    ax_arr, ay_arr = compute_acceleration(x, y)

    # --- Velocity update (Verlet) ---
    vx += 0.5 * (ax_old + ax_arr) * dt
    vy += 0.5 * (ay_old + ay_arr) * dt

    
    scat.set_offsets(np.column_stack((x, y)))
    ax.set_title(f"Frame {frame}", color='white', fontsize=10)

    return (scat,)

ani = FuncAnimation(
    fig,
    update,
    frames=900,
    interval=20,
    blit=True
)

HTML(ani.to_jshtml())